In [ ]:
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
import os

In [ ]:
# PDF indirme klasörünü ayarla
download_dir = "/Users/beyzaasan/Projects/HukukPusulasi/hukukPusulasi-veri/Kararlar"
os.makedirs(download_dir, exist_ok=True)

# Chrome ayarları
chrome_options = Options()
prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "plugins.always_open_pdf_externally": True
}
chrome_options.add_experimental_option("prefs", prefs)

# WebDriver'ı başlat
driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    # Siteye git
    driver.get("https://emsal.uyap.gov.tr")
    
    print("Sayfa yükleniyor...")
    time.sleep(3)
    
    # Detaylı Arama butonuna tıkla
    print("Detaylı Arama açılıyor...")
    try:
        detayli_arama = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, "Detaylı Arama")))
        detayli_arama.click()
        time.sleep(2)
    except:
        print("Detaylı Arama zaten açık olabilir, devam ediliyor...")
    
    # Aranacak Kelime alanına "tüketici" yaz
    print("'Tüketici' kelimesi giriliyor...")
    arama_input = wait.until(EC.presence_of_element_located((By.ID, "arananDetail")))
    
    # Scroll to element
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", arama_input)
    time.sleep(0.5)
    
    # JavaScript ile tıkla ve değer gir
    driver.execute_script("arguments[0].click();", arama_input)
    driver.execute_script("arguments[0].value = 'tüketici';", arama_input)
    
    # Input event'ini tetikle (Vue.js için)
    driver.execute_script("""
        var event = new Event('input', { bubbles: true });
        arguments[0].dispatchEvent(event);
    """, arama_input)
    time.sleep(1)
    
    # Hukuk Mahkemeleri dropdown'ını aç
    print("Hukuk Mahkemeleri seçiliyor...")
    
    # Bootstrap-select dropdown butonunu bul
    hukuk_dropdown_button = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "button[data-id='hukuk']")
    ))
    
    # Scroll ve tıkla
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", hukuk_dropdown_button)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", hukuk_dropdown_button)
    time.sleep(1)
    
    # "Hepsini Seç" butonunu bul ve tıkla
    print("Tüm seçenekler işaretleniyor...")
    hepsini_sec_button = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "button.bs-select-all")
    ))
    driver.execute_script("arguments[0].click();", hepsini_sec_button)
    time.sleep(1)
    
    # Dropdown'ı kapat (tekrar butona tıkla veya başka bir yere tıkla)
    driver.execute_script("arguments[0].click();", hukuk_dropdown_button)
    time.sleep(1)
    
    # Ara butonuna tıkla
    print("\nArama yapılıyor...")
    ara_button = wait.until(EC.presence_of_element_located((By.ID, "detaylıAramaG")))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", ara_button)
    time.sleep(1)
    
    # Butonun tıklanabilir olduğundan emin ol
    ara_button = wait.until(EC.element_to_be_clickable((By.ID, "detaylıAramaG")))
    driver.execute_script("arguments[0].click();", ara_button)
    
    print("Ara butonuna tıklandı, sonuçlar bekleniyor...")
    time.sleep(3)
    
    # Sonuçların yüklendiğini kontrol et - sayfa numarası veya kayıt sayısı görünene kadar bekle
    print("Sonuçlar yükleniyor...")
    try:
        # Toplam kayıt sayısını göster
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
        time.sleep(2)
        
        # Kaç kayıt bulunduğunu kontrol et
        try:
            kayit_sayisi = driver.find_element(By.XPATH, "//*[contains(text(), 'kayıt')]")
            print(f"✓ Sonuçlar yüklendi: {kayit_sayisi.text}")
        except:
            print("✓ Sonuçlar yüklendi (kayıt sayısı tespit edilemedi)")
    except:
        print("⚠ UYARI: Sonuçlar yüklenirken sorun olabilir!")
        print("Manuel kontrol için bekleyin...")
        input("Sonuçları tarayıcıda görebiliyor musunuz? Görebiliyorsanız ENTER'a basın...")
    
    # Sayfa başına sonuç sayısını 100 yap
    try:
        page_size_select = wait.until(EC.presence_of_element_located((
            By.XPATH,
            "//label/select | /html/body/div[2]/div/div[2]/div/div[2]/div/div/div/div[1]/div/div/div[2]/div[1]/div[1]/div/label/select"
        )))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", page_size_select)
        Select(page_size_select).select_by_visible_text("100")

        # Tablo yenilenmesini bekle
        tbody = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody")))
        old_first = tbody.find_elements(By.CSS_SELECTOR, "tr")[:1]
        if old_first:
            WebDriverWait(driver, 10).until(EC.staleness_of(old_first[0]))
        wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "table tbody tr")) >= 50)
        print("✓ Sayfa başına sonuç sayısı 100 olarak ayarlandı")
    except Exception as e:
        print(f"⚠ Sayfa başına sonuç sayısı ayarlanamadı: {e}")

    print("\n" + "="*50)
    print("PDF indirme işlemi başlıyor...")
    print("="*50 + "\n")
    
    current_page = 1
    total_downloaded = 0
    
    while True:
        print(f"\n=== Sayfa {current_page} işleniyor ===")
        
        # Sayfanın yüklenmesini bekle
        time.sleep(3)
        
        # Farklı tablo seçicilerini dene
        rows = []
        selectors = [
            "table tbody tr",
            "tbody tr",
            "tr[role='row']",
            ".table tbody tr",
            "#kararAlani tbody tr"
        ]
        
        for selector in selectors:
            rows = driver.find_elements(By.CSS_SELECTOR, selector)
            if len(rows) > 0:
                print(f"Satırlar bulundu (selector: {selector})")
                break
        
        if len(rows) == 0:
            print("HATA: Hiç satır bulunamadı!")
            print("\nSayfanın HTML yapısını kontrol etmek için manuel işlem gerekiyor.")
            print("Lütfen tarayıcıda sonuçları görebiliyor musunuz kontrol edin.")
            input("Devam etmek için ENTER'a basın (veya CTRL+C ile çıkın)...")
            break
        
        print(f"Toplam {len(rows)} kayıt bulundu")
        
        for i, row in enumerate(rows, 1):
            try:
                # 5. sütunu güvenli şekilde yakala (nth-child yerine index ve innerText kullan)
                cells = row.find_elements(By.CSS_SELECTOR, "td")
                if len(cells) >= 5:
                    status_cell = cells[4]

                    # Dinamik render için kısa bekleme
                    WebDriverWait(driver, 5).until(
                        lambda d: (status_cell.get_attribute("innerText") or status_cell.get_attribute("textContent") or "").strip() != ""
                    )

                    status = (status_cell.get_attribute("innerText") or status_cell.get_attribute("textContent") or "").strip()

                    # Hâlâ boşsa debug için HTML’i yazdırın
                    if not status:
                        print("DEBUG status_cell outerHTML:", status_cell.get_attribute("outerHTML"))

                    print(f"Kayıt {i}: Durum = {status}")
                    if status == "KESİNLEŞMEDİ":
                        print("  → Kesinleşmedi, atlanıyor...")
                        continue
                else:
                    print("Beklenen sütun sayısı yok, row HTML:", row.get_attribute("outerHTML"))
                    continue
                
                # Satıra tıkla (detay sayfasına gitmek için)
                row.click()
                time.sleep(2)  # Detay sayfasının yüklenmesini bekle
                
                # "PDF Olarak Kaydet" butonunu bul ve tıkla
                pdf_button = wait.until(EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, "a.btn.btn-light-primary[onclick*='kararSavePdf']")
                ))
                
                print(f"  → PDF indiriliyor...")
                pdf_button.click()
                
                # PDF'nin indirilmesini bekle
                time.sleep(3)
                total_downloaded += 1
                
                # Geri dön (liste sayfasına)
                driver.back()
                time.sleep(2)
                
                # Satırları yeniden bul (sayfa yenilendiği için)
                rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
                
            except Exception as e:
                print(f"  → Hata oluştu: {str(e)}")
                # Hata durumunda geri dön
                try:
                    driver.back()
                    time.sleep(2)
                except:
                    pass
                continue
        
        print(f"\nBu sayfada toplam {total_downloaded} PDF indirildi")
        
        # Sonraki sayfa var mı kontrol et (DataTables yapısına göre)
        try:
            # Mevcut ilk satırı referans al (staleness için)
            tbody = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody")))
            old_first_row = None
            try:
                old_first_row = tbody.find_elements(By.CSS_SELECTOR, "tr")[0]
            except Exception:
                pass

            # Next butonunu çeşitli seçicilerle bul
            next_button = None
            for selector in [
                "#detayAramaSonuclar_next",
                "#detayAramaSonuclar_paginate a.next",
                "#detayAramaSonuclar_paginate li.next a",
                "a.paginate_button.next",
            ]:
                buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                if buttons:
                    next_button = buttons[0]
                    break

            if not next_button:
                print("\nSonraki sayfa butonu bulunamadı, işlem tamamlandı.")
                break

            # Disabled kontrolü (buton veya ebeveyn li)
            parent_li = None
            try:
                parent_li = next_button.find_element(By.XPATH, "ancestor::li[1]")
            except Exception:
                parent_li = None
            classes = (next_button.get_attribute("class") or "") + " " + (parent_li.get_attribute("class") if parent_li else "")
            if "disabled" in classes:
                print("\nTüm sayfalar işlendi!")
                break

            # Tıkla ve tablo yenilenmesini bekle
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_button)
            driver.execute_script("arguments[0].click();", next_button)
            current_page += 1

            if old_first_row is not None:
                WebDriverWait(driver, 15).until(EC.staleness_of(old_first_row))
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
            print(f"→ Sonraki sayfaya geçildi: {current_page}")
            time.sleep(1)
            
        except Exception as e:
            print(f"\nSonraki sayfa bulunamadı veya işlem tamamlandı: {str(e)}")
            break
    
    print(f"\n✓ İşlem tamamlandı! Toplam {total_downloaded} PDF indirildi.")
    print(f"✓ PDF'ler şu klasöre kaydedildi: {download_dir}")

except Exception as e:
    print(f"Kritik hata: {str(e)}")

finally:
    input("\nKapatmak için ENTER'a basın...")
    driver.quit()

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.chains import QAGenerationChain
import os
import json

# OpenAI API anahtarınızı ayarlayın
os.environ["OPENAI_API_KEY"] = "YOUR-API-KEY"

# PDF'lerin bulunduğu dizin
pdf_dir = "downloaded_pdfs"  # Önceki hücrede kullanılan download_dir ile aynı olmalı

def generate_qa_pairs(pdf_path):
    # PDF'yi yükle ve metni çıkar
    loader = PyPDFLoader(pdf_path)
    pages = loader.load_and_split()
    
    # Metni bölümlere ayır
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    texts = text_splitter.split_documents(pages)
    
    # QA üretici zinciri oluştur
    chain = QAGenerationChain.from_llm(
        llm=ChatOpenAI(temperature=0.7)
    )
    
    qa_pairs = []
    
    # Her metin parçası için soru-cevap çiftleri üret
    for text in texts:
        result = chain.run(text.page_content)
        if isinstance(result, list):
            qa_pairs.extend(result)
        else:
            qa_pairs.append(result)
    
    return qa_pairs

# Tüm PDF'ler için QA çiftleri üret
all_qa_pairs = []
for filename in os.listdir(pdf_dir):
    if filename.endswith('.pdf'):
        pdf_path = os.path.join(pdf_dir, filename)
        print(f"Processing {filename}...")
        
        try:
            qa_pairs = generate_qa_pairs(pdf_path)
            all_qa_pairs.extend(qa_pairs)
            print(f"Generated {len(qa_pairs)} QA pairs from {filename}")
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")

# Sonuçları JSON dosyasına kaydet
output_file = "mahkeme_kararlari_qa.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_qa_pairs, f, ensure_ascii=False, indent=2)

print(f"\nToplam {len(all_qa_pairs)} soru-cevap çifti üretildi.")
print(f"Sonuçlar {output_file} dosyasına kaydedildi.")

In [4]:
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
import os
import json
from datetime import datetime

# Yapılandırma
download_dir = "/Users/beyzaasan/Projects/HukukPusulasi/hukukPusulasi-veri/Kararlar"
progress_file = "indirme_ilerleme.json"
max_retries = 3
retry_delay = 5

os.makedirs(download_dir, exist_ok=True)

def load_progress():
    """İlerleme dosyasını yükle"""
    if os.path.exists(progress_file):
        with open(progress_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {"downloaded_files": [], "last_page": 0, "last_row": 0, "total_downloaded": 0}

def save_progress(progress):
    """İlerlemeyi kaydet"""
    with open(progress_file, 'w', encoding='utf-8') as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)

def get_pdf_filename():
    """Son indirilen PDF'in adını al"""
    files = [f for f in os.listdir(download_dir) if f.endswith('.pdf')]
    if not files:
        return None
    latest = max([os.path.join(download_dir, f) for f in files], key=os.path.getmtime)
    return os.path.basename(latest)

def init_driver():
    """WebDriver'ı başlat"""
    chrome_options = Options()
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "plugins.always_open_pdf_externally": True
    }
    chrome_options.add_experimental_option("prefs", prefs)
    driver = webdriver.Chrome(options=chrome_options)
    return driver, WebDriverWait(driver, 10)

def perform_search(driver, wait):
    """Arama işlemini gerçekleştir"""
    driver.get("https://emsal.uyap.gov.tr")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Sayfa yükleniyor...")
    time.sleep(3)
    
    try:
        detayli_arama = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, "Detaylı Arama")))
        detayli_arama.click()
        time.sleep(2)
    except:
        print("Detaylı Arama zaten açık...")
    
    arama_input = wait.until(EC.presence_of_element_located((By.ID, "arananDetail")))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", arama_input)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", arama_input)
    driver.execute_script("arguments[0].value = 'tüketici';", arama_input)
    driver.execute_script("""
        var event = new Event('input', { bubbles: true });
        arguments[0].dispatchEvent(event);
    """, arama_input)
    time.sleep(1)
    
    hukuk_dropdown_button = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "button[data-id='hukuk']")
    ))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", hukuk_dropdown_button)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", hukuk_dropdown_button)
    time.sleep(1)
    
    hepsini_sec_button = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "button.bs-select-all")
    ))
    driver.execute_script("arguments[0].click();", hepsini_sec_button)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", hukuk_dropdown_button)
    time.sleep(1)
    
    ara_button = wait.until(EC.element_to_be_clickable((By.ID, "detaylıAramaG")))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", ara_button)
    time.sleep(1)
    driver.execute_script("arguments[0].click();", ara_button)
    time.sleep(3)
    
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
    time.sleep(2)
    
    try:
        page_size_select = wait.until(EC.presence_of_element_located((
            By.XPATH, "//label/select"
        )))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", page_size_select)
        Select(page_size_select).select_by_visible_text("100")
        time.sleep(3)
        print("✓ Sayfa başına 100 sonuç ayarlandı")
    except:
        print("⚠ Sayfa boyutu ayarlanamadı")

def navigate_to_page(driver, wait, target_page):
    """Belirli bir sayfaya git"""
    if target_page <= 1:
        return True
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Sayfa {target_page}'ye gidiliyor...")
    for page in range(2, target_page + 1):
        try:
            tbody = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody")))
            old_first_row = tbody.find_elements(By.CSS_SELECTOR, "tr")[0]
            
            next_button = None
            for selector in [
                "#detayAramaSonuclar_next",
                "#detayAramaSonuclar_paginate a.next",
                "a.paginate_button.next",
            ]:
                buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                if buttons:
                    next_button = buttons[0]
                    break
            
            if not next_button:
                return False
            
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_button)
            driver.execute_script("arguments[0].click();", next_button)
            
            WebDriverWait(driver, 15).until(EC.staleness_of(old_first_row))
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
            time.sleep(1)
            
        except Exception as e:
            print(f"⚠ Sayfa {page}'ye giderken hata: {str(e)}")
            return False
    
    return True

def download_pdf_with_retry(driver, wait, row, row_index):
    """PDF indirmeyi yeniden deneme mekanizmasıyla gerçekleştir"""
    for attempt in range(max_retries):
        try:
            row.click()
            time.sleep(2)
            
            pdf_button = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "a.btn.btn-light-primary[onclick*='kararSavePdf']")
            ))
            
            old_files = set(os.listdir(download_dir))
            pdf_button.click()
            time.sleep(3)
            
            # Yeni dosya oluştuğunu kontrol et
            for _ in range(10):
                new_files = set(os.listdir(download_dir)) - old_files
                if new_files:
                    driver.back()
                    time.sleep(2)
                    return list(new_files)[0]
                time.sleep(1)
            
            driver.back()
            time.sleep(2)
            return None
            
        except Exception as e:
            print(f"  ⚠ Deneme {attempt + 1}/{max_retries} başarısız: {str(e)}")
            try:
                driver.back()
                time.sleep(2)
                # Satırları yeniden bul
                rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
                if row_index < len(rows):
                    row = rows[row_index]
            except:
                pass
            
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                return None
    
    return None

def main():
    progress = load_progress()
    print(f"\n{'='*60}")
    print(f"İndirme devam ediyor...")
    print(f"Son durum: Sayfa {progress['last_page']}, Satır {progress['last_row']}")
    print(f"Toplam indirilen: {progress['total_downloaded']} PDF")
    print(f"{'='*60}\n")
    
    driver = None
    try:
        driver, wait = init_driver()
        perform_search(driver, wait)
        
        # Kaldığı sayfaya git
        if progress['last_page'] > 1:
            if not navigate_to_page(driver, wait, progress['last_page']):
                print("⚠ Sayfaya gidilemedi, baştan başlanıyor...")
                progress['last_page'] = 1
                progress['last_row'] = 0
        
        current_page = progress['last_page'] if progress['last_page'] > 0 else 1
        
        while True:
            print(f"\n[{datetime.now().strftime('%H:%M:%S')}] === Sayfa {current_page} işleniyor ===")
            time.sleep(3)
            
            rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
            if not rows:
                print("⚠ Satır bulunamadı!")
                break
            
            print(f"Toplam {len(rows)} kayıt bulundu")
            
            # Kaldığı satırdan devam et
            start_row = progress['last_row'] if current_page == progress['last_page'] else 0
            
            for i in range(start_row, len(rows)):
                row = rows[i]
                try:
                    cells = row.find_elements(By.CSS_SELECTOR, "td")
                    if len(cells) >= 5:
                        status_cell = cells[4]
                        WebDriverWait(driver, 5).until(
                            lambda d: (status_cell.get_attribute("innerText") or "").strip() != ""
                        )
                        status = (status_cell.get_attribute("innerText") or "").strip()
                        
                        print(f"[{i+1}/{len(rows)}] Durum: {status}")
                        
                        if status == "KESİNLEŞMEDİ":
                            print("  → Kesinleşmedi, atlanıyor...")
                            continue
                        
                        filename = download_pdf_with_retry(driver, wait, row, i)
                        
                        if filename:
                            progress['total_downloaded'] += 1
                            progress['downloaded_files'].append(filename)
                            progress['last_page'] = current_page
                            progress['last_row'] = i + 1
                            save_progress(progress)
                            print(f"  ✓ İndirildi: {filename} (Toplam: {progress['total_downloaded']})")
                        else:
                            print(f"  ✗ İndirilemedi, devam ediliyor...")
                        
                        # Satırları yeniden bul
                        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
                        
                except Exception as e:
                    print(f"  ✗ Hata: {str(e)}, devam ediliyor...")
                    try:
                        driver.back()
                        time.sleep(2)
                        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
                    except:
                        pass
                    continue
            
            # Sonraki sayfaya geç
            try:
                tbody = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody")))
                old_first_row = tbody.find_elements(By.CSS_SELECTOR, "tr")[0]
                
                next_button = None
                for selector in [
                    "#detayAramaSonuclar_next",
                    "#detayAramaSonuclar_paginate a.next",
                    "a.paginate_button.next",
                ]:
                    buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                    if buttons:
                        next_button = buttons[0]
                        break
                
                if not next_button:
                    print("\n✓ Tüm sayfalar tamamlandı!")
                    break
                
                classes = next_button.get_attribute("class") or ""
                if "disabled" in classes:
                    print("\n✓ Tüm sayfalar tamamlandı!")
                    break
                
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_button)
                driver.execute_script("arguments[0].click();", next_button)
                current_page += 1
                progress['last_page'] = current_page
                progress['last_row'] = 0
                save_progress(progress)
                
                WebDriverWait(driver, 15).until(EC.staleness_of(old_first_row))
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
                time.sleep(1)
                
            except Exception as e:
                print(f"\n⚠ Sayfa geçişi başarısız: {str(e)}")
                break
        
        print(f"\n{'='*60}")
        print(f"✓ İşlem tamamlandı!")
        print(f"✓ Toplam {progress['total_downloaded']} PDF indirildi")
        print(f"✓ Dosyalar: {download_dir}")
        print(f"{'='*60}\n")
        
    except KeyboardInterrupt:
        print("\n\n⚠ Kullanıcı tarafından durduruldu")
        print(f"İlerleme kaydedildi. Toplam: {progress['total_downloaded']} PDF")
        print("Scripti tekrar çalıştırarak kaldığı yerden devam edebilirsiniz.")
        
    except Exception as e:
        print(f"\n⚠ Kritik hata: {str(e)}")
        print(f"İlerleme kaydedildi. Scripti tekrar çalıştırın.")
        
    finally:
        if driver:
            driver.quit()
        save_progress(progress)

if __name__ == "__main__":
    main()


İndirme devam ediyor...
Son durum: Sayfa 6, Satır 83
Toplam indirilen: 179 PDF

[19:35:34] Sayfa yükleniyor...
✓ Sayfa başına 100 sonuç ayarlandı
[19:35:54] Sayfa 6'ye gidiliyor...

[19:36:02] === Sayfa 6 işleniyor ===
Toplam 100 kayıt bulundu
[84/100] Durum: KESİNLEŞTİ
  ✗ İndirilemedi, devam ediliyor...
[85/100] Durum: KESİNLEŞTİ
  ✓ İndirildi: 2025_44 (3).pdf (Toplam: 180)
[86/100] Durum: KESİNLEŞMEDİ
  → Kesinleşmedi, atlanıyor...
[87/100] Durum: KESİNLEŞTİ
  ✓ İndirildi: 2025_40.pdf (Toplam: 181)
[88/100] Durum: KESİNLEŞMEDİ
  → Kesinleşmedi, atlanıyor...
[89/100] Durum: KESİNLEŞMEDİ
  → Kesinleşmedi, atlanıyor...
[90/100] Durum: KESİNLEŞTİ
  ✓ İndirildi: 2025_39.pdf (Toplam: 182)
[91/100] Durum: KESİNLEŞMEDİ
  → Kesinleşmedi, atlanıyor...
[92/100] Durum: KESİNLEŞMEDİ
  → Kesinleşmedi, atlanıyor...
[93/100] Durum: KESİNLEŞMEDİ
  → Kesinleşmedi, atlanıyor...
[94/100] Durum: KESİNLEŞTİ
  ✓ İndirildi: 2025_35.pdf (Toplam: 183)
[95/100] Durum: KESİNLEŞMEDİ
  → Kesinleşmedi, atlanıyor